# Detecção de Fraude em Cartões de Crédito 💳

Projeto de classificação para detectar transações fraudulentas em um dataset extremamente desbalanceado.

## Objetivos

- Explorar os dados e medir o desbalanceamento das classes;
- Criar e transformar variáveis de entrada;
- Separar os dados em treino, validação e teste, preservando a proporção de fraudes;
- Padronizar os dados sem vazamento de informação (data leakage);
- Comparar Regressão Logística, Random Forest e XGBoost;
- Testar undersampling e SMOTE como alternativas de balanceamento;
- Avaliar principalmente Precision, Recall e F1 da classe de fraude;
- Analisar as curvas ROC e Precision-Recall;
- Escolher o threshold do XGBoost na validação, sem usar o conjunto de teste para ajuste;
- Usar SHAP para explicar o comportamento global e uma decisão individual do modelo.

> **Importante:** o dataset é carregado diretamente pela URL. O arquivo `creditcard.csv` não é salvo no repositório.

## 1. Importação das bibliotecas e configurações

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as xgb

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
FIG_DIR = Path("figuras")
FIG_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

print("Bibliotecas carregadas com sucesso.")

## 2. Carregamento do dataset

A base utilizada possui as variáveis `V1` a `V28`, além de `Time`, `Amount` e `Class`. `Class = 0` representa uma transação normal e `Class = 1` uma fraude.

In [ ]:
DATASET_URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"

df = pd.read_csv(DATASET_URL)

print(f"Formato do dataset: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print("\nPrimeiras linhas:")
display(df.head())

## 3. Exploração inicial

Antes de treinar os modelos, é importante conhecer a estrutura da base, verificar valores ausentes/duplicados e quantificar o desbalanceamento.

In [ ]:
print("Tipos das colunas:")
display(df.dtypes.to_frame("dtype"))

print("\nValores ausentes por coluna:")
display(df.isna().sum().sort_values(ascending=False).head(10).to_frame("missing"))

print(f"\nLinhas duplicadas: {df.duplicated().sum():,}")

class_counts = df["Class"].value_counts().sort_index()
class_percent = df["Class"].value_counts(normalize=True).sort_index() * 100

resumo_classes = pd.DataFrame({
    "quantidade": class_counts,
    "percentual": class_percent,
})
resumo_classes.index = ["Normal (0)", "Fraude (1)"]

display(resumo_classes)

print(f"\nTotal de fraudes: {int(class_counts.loc[1]):,}")
print(f"Percentual de fraudes: {class_percent.loc[1]:.4f}%")

In [ ]:
counts = df["Class"].value_counts().sort_index()
labels = ["Normal (0)", "Fraude (1)"]

plt.figure(figsize=(7, 4))
plt.bar(labels, [counts.loc[0], counts.loc[1]])
plt.title("Distribuição das classes")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.savefig(FIG_DIR / "distribuicao_classes.png", dpi=160, bbox_inches="tight")
plt.show()

### Por que a acurácia pode enganar?

Como a fraude é uma classe muito rara, um modelo que previsse praticamente todas as transações como normais poderia apresentar acurácia elevada mesmo sem cumprir o objetivo do problema. Por isso, este projeto acompanha especialmente **precision, recall e F1 da classe 1 (fraude)**.

## 4. Preparação e feature engineering

A transformação `log1p` reduz o efeito da assimetria do valor das transações. Em seguida, os dados serão divididos em **treino, validação e teste**.

A ordem é importante: o `StandardScaler` será ajustado (`fit`) somente no conjunto de treino. Depois, ele será usado para transformar validação e teste. Assim evitamos data leakage.

In [ ]:
df_model = df.copy()

df_model["Log_Amount"] = np.log1p(df_model["Amount"])

feature_columns = ["Time", "Log_Amount"] + [f"V{i}" for i in range(1, 29)]
X = df_model[feature_columns].copy()
y = df_model["Class"].astype(int).copy()

# 60% treino, 20% validação e 20% teste.
X_temp, X_test_raw, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

print("Tamanho dos conjuntos:")
print(f"Treino:      {len(X_train_raw):,} ({len(X_train_raw)/len(X):.1%})")
print(f"Validação:   {len(X_val_raw):,} ({len(X_val_raw)/len(X):.1%})")
print(f"Teste:       {len(X_test_raw):,} ({len(X_test_raw)/len(X):.1%})")

print("\nFraudes em cada conjunto:")
print(f"Treino:      {int(y_train.sum())}")
print(f"Validação:   {int(y_val.sum())}")
print(f"Teste:       {int(y_test.sum())}")

In [ ]:
scaler = StandardScaler()

X_train = pd.DataFrame(
    scaler.fit_transform(X_train_raw),
    columns=X_train_raw.columns,
    index=X_train_raw.index,
)

X_val = pd.DataFrame(
    scaler.transform(X_val_raw),
    columns=X_val_raw.columns,
    index=X_val_raw.index,
)

X_test = pd.DataFrame(
    scaler.transform(X_test_raw),
    columns=X_test_raw.columns,
    index=X_test_raw.index,
)

print("StandardScaler ajustado somente no conjunto de treino.")

## 5. Estratégias de desbalanceamento

Serão testadas duas estratégias clássicas apenas sobre o conjunto de treino:

- **Undersampling:** reduz exemplos da classe majoritária;
- **SMOTE:** cria exemplos sintéticos da classe minoritária.

O conjunto de validação e o conjunto de teste permanecem com a distribuição original, porque são usados para avaliar o comportamento do modelo em um cenário próximo do problema real.

In [ ]:
rus = RandomUnderSampler(random_state=RANDOM_STATE)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

balanceamento = pd.DataFrame({
    "Estratégia": ["Original", "Undersampling", "SMOTE"],
    "Classe 0": [
        int((y_train == 0).sum()),
        int((y_train_rus == 0).sum()),
        int((y_train_smote == 0).sum()),
    ],
    "Classe 1": [
        int((y_train == 1).sum()),
        int((y_train_rus == 1).sum()),
        int((y_train_smote == 1).sum()),
    ],
})

display(balanceamento)

## 6. Funções auxiliares para avaliação

In [ ]:
def extrair_metricas(y_true, y_pred, y_prob, nome_modelo, threshold=0.5):
    relatorio = classification_report(
        y_true,
        y_pred,
        output_dict=True,
        zero_division=0,
    )

    return {
        "Modelo": nome_modelo,
        "Threshold": threshold,
        "Precision (fraude)": relatorio["1"]["precision"],
        "Recall (fraude)": relatorio["1"]["recall"],
        "F1 (fraude)": relatorio["1"]["f1-score"],
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC / Average Precision": average_precision_score(y_true, y_prob),
    }


def encontrar_threshold_max_f1(y_true, y_prob):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)

    f1_scores = (
        2 * precisions[:-1] * recalls[:-1]
        / (precisions[:-1] + recalls[:-1] + 1e-12)
    )

    best_idx = int(np.nanargmax(f1_scores))
    best_threshold = float(thresholds[best_idx])

    return best_threshold, precisions, recalls, thresholds, f1_scores

## 7. Regressão Logística — baseline

A Regressão Logística será usada como modelo de referência. Para o baseline principal, usamos `class_weight="balanced"`, permitindo que o algoritmo dê mais peso à classe rara sem alterar a distribuição dos dados de treino.

Também serão comparadas duas variações com os dados balanceados por undersampling e SMOTE.

In [ ]:
modelos = {}

lr_balanced = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
lr_balanced.fit(X_train, y_train)
modelos["Regressão Logística (class_weight)"] = lr_balanced

lr_rus = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)
lr_rus.fit(X_train_rus, y_train_rus)
modelos["Regressão Logística (undersampling)"] = lr_rus

lr_smote = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)
lr_smote.fit(X_train_smote, y_train_smote)
modelos["Regressão Logística (SMOTE)"] = lr_smote

print("Modelos de Regressão Logística treinados.")

## 8. Random Forest

O Random Forest também será treinado usando pesos de classe. O conjunto original de treino é mantido, evitando que a avaliação seja feita sobre dados artificialmente balanceados.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)
modelos["Random Forest (class_weight)"] = rf_model

print("Random Forest treinado.")

## 9. XGBoost

No XGBoost, `scale_pos_weight` é calculado a partir da proporção entre exemplos negativos e positivos no treino. O modelo usa a distribuição original de treino, compensando o desbalanceamento durante o aprendizado.

In [ ]:
negativos = int((y_train == 0).sum())
positivos = int((y_train == 1).sum())
scale_pos_weight = negativos / positivos

xgb_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_model.fit(X_train, y_train)
modelos["XGBoost (scale_pos_weight)"] = xgb_model

print(f"scale_pos_weight = {scale_pos_weight:.2f}")
print("XGBoost treinado.")

## 10. Comparação dos modelos na validação

A validação é usada para comparar os modelos e fazer ajustes. O teste permanece intocado até a avaliação final.

In [ ]:
resultados_validacao = []
probabilidades_validacao = {}

for nome, modelo in modelos.items():
    prob = modelo.predict_proba(X_val)[:, 1]
    pred = (prob >= 0.5).astype(int)
    probabilidades_validacao[nome] = prob
    resultados_validacao.append(
        extrair_metricas(y_val, pred, prob, nome, threshold=0.5)
    )

resultados_validacao_df = (
    pd.DataFrame(resultados_validacao)
    .sort_values("Recall (fraude)", ascending=False)
    .reset_index(drop=True)
)

display(resultados_validacao_df)

## 11. Curva ROC e curva Precision-Recall

As duas curvas ajudam a visualizar o comportamento dos classificadores em diferentes limiares. Para um problema tão desbalanceado, a curva Precision-Recall recebe atenção especial porque evidencia melhor a relação entre capturar fraudes e gerar falsos positivos.

In [ ]:
modelos_principais = [
    "Regressão Logística (class_weight)",
    "Random Forest (class_weight)",
    "XGBoost (scale_pos_weight)",
]

plt.figure(figsize=(8, 5))
for nome in modelos_principais:
    prob = probabilidades_validacao[nome]
    fpr, tpr, _ = roc_curve(y_val, prob)
    auc_roc = roc_auc_score(y_val, prob)
    plt.plot(fpr, tpr, label=f"{nome} (AUC={auc_roc:.4f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Aleatório")
plt.xlabel("Taxa de falsos positivos")
plt.ylabel("Taxa de verdadeiros positivos (Recall)")
plt.title("Curva ROC — conjunto de validação")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "curva_roc_validacao.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
for nome in modelos_principais:
    prob = probabilidades_validacao[nome]
    precision, recall, _ = precision_recall_curve(y_val, prob)
    ap = average_precision_score(y_val, prob)
    plt.plot(recall, precision, label=f"{nome} (AP={ap:.4f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision-Recall — conjunto de validação")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "curva_precision_recall_validacao.png", dpi=160, bbox_inches="tight")
plt.show()

## 12. Ajuste do threshold do XGBoost

O valor padrão de 0.5 não precisa ser o melhor para um problema de fraude. Neste projeto, o threshold é escolhido **somente com o conjunto de validação**, buscando o maior F1 da classe de fraude.

Depois de escolhido, o valor é aplicado uma única vez ao conjunto de teste para obter a avaliação final.

In [ ]:
y_prob_xgb_val = probabilidades_validacao["XGBoost (scale_pos_weight)"]

threshold_xgb, precisions, recalls, thresholds, f1_scores = encontrar_threshold_max_f1(
    y_val,
    y_prob_xgb_val,
)

print(f"Threshold escolhido na validação (maior F1): {threshold_xgb:.6f}")
print(f"F1 máximo na validação: {f1_scores.max():.4f}")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(thresholds, precisions[:-1], label="Precision")
plt.plot(thresholds, recalls[:-1], label="Recall")
plt.plot(thresholds, f1_scores, label="F1")
plt.axvline(
    threshold_xgb,
    linestyle="--",
    label=f"Threshold escolhido = {threshold_xgb:.4f}",
)
plt.xlabel("Threshold")
plt.ylabel("Métrica")
plt.title("Precision, Recall e F1 em função do threshold — validação")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "threshold_xgboost_validacao.png", dpi=160, bbox_inches="tight")
plt.show()

## 13. Avaliação final no conjunto de teste

Agora que modelo e threshold foram definidos usando treino + validação, o conjunto de teste será usado para a avaliação final.

In [ ]:
y_prob_test = xgb_model.predict_proba(X_test)[:, 1]

y_pred_xgb_05 = (y_prob_test >= 0.5).astype(int)
y_pred_xgb_threshold = (y_prob_test >= threshold_xgb).astype(int)

resultado_xgb_05 = extrair_metricas(
    y_test,
    y_pred_xgb_05,
    y_prob_test,
    "XGBoost — teste (threshold 0.5)",
    threshold=0.5,
)

resultado_xgb_threshold = extrair_metricas(
    y_test,
    y_pred_xgb_threshold,
    y_prob_test,
    "XGBoost — teste (threshold ajustado)",
    threshold=threshold_xgb,
)

comparacao_threshold = pd.DataFrame([
    resultado_xgb_05,
    resultado_xgb_threshold,
])

display(comparacao_threshold)

In [ ]:
print("Relatório de classificação — XGBoost com threshold ajustado")
print(classification_report(y_test, y_pred_xgb_threshold, digits=4, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred_xgb_threshold)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm)
ax.set_xticks([0, 1], labels=["Normal", "Fraude"])
ax.set_yticks([0, 1], labels=["Normal", "Fraude"])
ax.set_xlabel("Predito")
ax.set_ylabel("Verdadeiro")
ax.set_title("Matriz de confusão — XGBoost com threshold ajustado")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(FIG_DIR / "matriz_confusao_xgboost.png", dpi=160, bbox_inches="tight")
plt.show()

## 14. Curvas finais do XGBoost no conjunto de teste

In [ ]:
fpr_test, tpr_test, _ = roc_curve(y_test, y_prob_test)
precision_test, recall_test, _ = precision_recall_curve(y_test, y_prob_test)

roc_auc_test = roc_auc_score(y_test, y_prob_test)
ap_test = average_precision_score(y_test, y_prob_test)

plt.figure(figsize=(8, 5))
plt.plot(fpr_test, tpr_test, label=f"XGBoost (ROC-AUC={roc_auc_test:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Aleatório")
plt.xlabel("Taxa de falsos positivos")
plt.ylabel("Recall")
plt.title("Curva ROC — XGBoost no teste")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "curva_roc_xgboost_teste.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(recall_test, precision_test, label=f"XGBoost (AP={ap_test:.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision-Recall — XGBoost no teste")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "curva_precision_recall_xgboost_teste.png", dpi=160, bbox_inches="tight")
plt.show()

## 15. SHAP — explicabilidade global

O SHAP ajuda a explicar quanto cada variável contribuiu para a saída do modelo.

Primeiro será analisada uma amostra do conjunto de teste para identificar, de forma global, quais variáveis tiveram maior impacto absoluto nas previsões do XGBoost.

In [ ]:
shap_sample_size = min(500, len(X_test))
X_shap_sample = X_test.sample(shap_sample_size, random_state=RANDOM_STATE)

explainer = shap.TreeExplainer(xgb_model)
shap_sample_explanation = explainer(X_shap_sample)

plt.figure()
shap.summary_plot(
    shap_sample_explanation.values,
    X_shap_sample,
    show=False,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_summary.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Resumo em barras: impacto absoluto médio de cada variável.
plt.figure()
shap.summary_plot(
    shap_sample_explanation.values,
    X_shap_sample,
    plot_type="bar",
    show=False,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_importancia_global.png", dpi=160, bbox_inches="tight")
plt.show()

importancias_shap = pd.DataFrame({
    "feature": X_shap_sample.columns,
    "mean_abs_shap": np.abs(shap_sample_explanation.values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

display(importancias_shap.head(10))

## 16. SHAP — explicação de uma transação individual

Para demonstrar uma explicação local, será selecionada uma transação que o modelo marcou como fraude usando o threshold ajustado. Quando houver mais de uma, a primeira encontrada será usada; caso não haja, usamos a transação de maior probabilidade prevista como fallback.

In [ ]:
teste_indices = np.where(y_pred_xgb_threshold == 1)[0]

if len(teste_indices) > 0:
    indice_local = int(teste_indices[0])
else:
    indice_local = int(np.argmax(y_prob_test))

X_individual = X_test.iloc[[indice_local]]
prob_individual = float(y_prob_test[indice_local])
classe_real_individual = int(y_test.iloc[indice_local])
classe_predita_individual = int(y_pred_xgb_threshold[indice_local])

print(f"Índice da transação analisada: {X_individual.index[0]}")
print(f"Classe real: {classe_real_individual}")
print(f"Probabilidade de fraude: {prob_individual:.6f}")
print(f"Classe prevista com threshold ajustado: {classe_predita_individual}")

In [ ]:
shap_individual = explainer(X_individual)

plt.figure(figsize=(10, 6))
shap.plots.waterfall(
    shap_individual[0],
    max_display=12,
    show=False,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_waterfall_transacao.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
contribuicoes_individuais = pd.DataFrame({
    "feature": X_individual.columns,
    "shap_value": shap_individual.values[0],
})
contribuicoes_individuais["abs_shap"] = contribuicoes_individuais["shap_value"].abs()
contribuicoes_individuais["direcao"] = np.where(
    contribuicoes_individuais["shap_value"] >= 0,
    "Aumenta a saída do modelo para fraude",
    "Reduz a saída do modelo para fraude",
)

contribuicoes_individuais = (
    contribuicoes_individuais
    .sort_values("abs_shap", ascending=False)
    .reset_index(drop=True)
)

display(contribuicoes_individuais.head(10))

## 17. Comparação final no conjunto de teste

Para a comparação principal, os modelos são avaliados no mesmo conjunto de teste, usando threshold 0.5. Em seguida, o XGBoost também é apresentado com o threshold ajustado na validação.

In [ ]:
resultados_teste = []

for nome, modelo in modelos.items():
    prob = modelo.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    resultados_teste.append(
        extrair_metricas(y_test, pred, prob, nome, threshold=0.5)
    )

resultados_teste_df = (
    pd.DataFrame(resultados_teste)
    .sort_values("Recall (fraude)", ascending=False)
    .reset_index(drop=True)
)

display(resultados_teste_df)

resultados_com_threshold = pd.concat(
    [
        resultados_teste_df,
        pd.DataFrame([resultado_xgb_threshold]),
    ],
    ignore_index=True,
)

display(resultados_com_threshold)

resultados_teste_df.to_csv("resultados_modelos.csv", index=False)
comparacao_threshold.to_csv("comparacao_threshold_xgboost.csv", index=False)

print("Arquivos salvos: resultados_modelos.csv e comparacao_threshold_xgboost.csv")

## 18. Conclusões

O ponto central deste projeto não é buscar a maior acurácia, mas avaliar se o modelo identifica a classe rara de fraude de forma útil. Para isso, a análise considera principalmente **recall, precision e F1 da classe 1**.

O threshold do XGBoost foi escolhido na validação pelo maior F1, e somente depois foi aplicado ao conjunto de teste. Assim, o conjunto de teste não participa da decisão de ajuste.

A análise SHAP complementa as métricas: a visão global mostra quais variáveis mais influenciam o modelo em média, enquanto a visão individual mostra quais características contribuíram para uma previsão específica.

## 19. Geração automática do README com os resultados reais

A última célula cria `README_final.md` com os números obtidos na execução do notebook. Isso evita colocar resultados inventados ou fixos antes de executar o modelo.

In [ ]:
linhas_tabela = []
for _, row in resultados_com_threshold.iterrows():
    linhas_tabela.append(
        f"| {row['Modelo']} | {row['Precision (fraude)']:.4f} | "
        f"{row['Recall (fraude)']:.4f} | {row['F1 (fraude)']:.4f} | "
        f"{row['ROC-AUC']:.4f} | {row['PR-AUC / Average Precision']:.4f} |"
    )

top_features = importancias_shap.head(5)["feature"].tolist()
contrib_top = contribuicoes_individuais.head(5)
contrib_texto = "\n".join(
    f"- **{row['feature']}**: {row['direcao']} (SHAP = {row['shap_value']:.4f})"
    for _, row in contrib_top.iterrows()
)

readme_final = """# Detecção de Fraude em Cartões de Crédito 💳

Projeto de Machine Learning para detecção de fraude em transações de cartão de crédito, desenvolvido como parte de um desafio da DIO.

## 📌 O problema

O dataset apresenta um desbalanceamento extremo: a classe de fraude representa uma pequena parcela das transações. Nesse cenário, a acurácia isoladamente pode criar uma impressão enganosa de desempenho.

Por isso, este projeto dá atenção especial às métricas da classe de fraude (`Class = 1`): **Precision, Recall e F1-score**.

- **Precision:** entre as transações marcadas como fraude, quantas realmente eram fraude.
- **Recall:** entre as fraudes reais, quantas foram identificadas pelo modelo.
- **F1-score:** média harmônica entre precision e recall.

## 🧹 Preparação dos dados

- Dataset carregado diretamente pela URL, sem armazenar `creditcard.csv` no repositório;
- Criação de `Log_Amount` com `np.log1p` para reduzir a assimetria do valor das transações;
- Divisão estratificada em treino, validação e teste;
- `StandardScaler` ajustado somente no conjunto de treino, evitando data leakage;
- Undersampling e SMOTE testados somente no conjunto de treino.

## 🤖 Modelos

Foram comparados:

1. Regressão Logística com `class_weight="balanced"`;
2. Regressão Logística com undersampling;
3. Regressão Logística com SMOTE;
4. Random Forest com `class_weight="balanced_subsample"`;
5. XGBoost com `scale_pos_weight`.

## 📊 Resultados no conjunto de teste

Os resultados abaixo foram gerados automaticamente pela execução deste notebook.

| Modelo | Precision (fraude) | Recall (fraude) | F1 (fraude) | ROC-AUC | PR-AUC |
|---|---:|---:|---:|---:|---:|
""" + "\n".join(linhas_tabela) + """

## 🎚️ Threshold do XGBoost

O threshold padrão de 0.5 foi comparado com um threshold ajustado. O valor ajustado foi escolhido **no conjunto de validação**, maximizando o F1 da classe de fraude.

**Threshold escolhido:** `""" + f"{threshold_xgb:.6f}" + "`\n\n" + """

No conjunto de teste, a comparação foi:

| Configuração | Precision | Recall | F1 |
|---|---:|---:|---:|
| Threshold 0.5 | """ + f"{resultado_xgb_05['Precision (fraude)']:.4f} | {resultado_xgb_05['Recall (fraude)']:.4f} | {resultado_xgb_05['F1 (fraude)']:.4f}" + " |\n" + "| Threshold ajustado | """ + f"{resultado_xgb_threshold['Precision (fraude)']:.4f} | {resultado_xgb_threshold['Recall (fraude)']:.4f} | {resultado_xgb_threshold['F1 (fraude)']:.4f}" + " |\n\n" + """
## 🔍 SHAP

Na análise global, as cinco variáveis com maior impacto absoluto médio foram:

""" + "\n".join(f"- `{feature}`" for feature in top_features) + "\n\n" + """
Para uma transação marcada como fraude pelo threshold ajustado, as maiores contribuições individuais foram:

""" + contrib_texto + "\n\n" + """
## 🚀 O que foi acrescentado/adaptado no projeto

- Separação explícita entre treino, validação e teste;
- Padronização sem usar informações do conjunto de teste;
- Comparação de undersampling e SMOTE;
- Inclusão de Random Forest e XGBoost;
- Ajuste de threshold baseado na validação, evitando usar o teste para escolher o limiar;
- Avaliação por Precision, Recall, F1, ROC-AUC e PR-AUC;
- SHAP global e explicação individual de uma transação;
- Salvamento das principais tabelas e figuras geradas pelo notebook.

## 📁 Estrutura do repositório

```text
.
├── deteccao_fraude_cartoes.ipynb
├── README.md
├── requisitos.txt
├── resultados_modelos.csv
├── comparacao_threshold_xgboost.csv
└── figuras/
```

## ✅ Observações

O dataset não é incluído no repositório. O notebook faz o carregamento pela URL.

As métricas e análises deste README correspondem à execução do notebook e devem ser mantidas junto com as saídas salvas, para que o repositório tenha evidência reproduzível do resultado.
"""

Path("README_final.md").write_text(readme_final, encoding="utf-8")
print("README_final.md gerado com os resultados reais da execução.")